# Model comparison: machine learning models

SVM-RBF, Random Forest, XGBoost, Extra Trees and Stacking-Tree on all 40 eye and head features
under cross-user mixing with stratified 10-fold cross-validation.

Uses `../data/window_features.csv`. Fold results and means are saved in `results/`.

In [ ]:
import os
import time
import warnings
from collections import OrderedDict
from inspect import signature
from time import perf_counter

import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

DATA_PATH = os.path.join("..", "data", "window_features.csv")
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

df = pd.read_csv(DATA_PATH)
df = df.dropna(how="all")

exp_feature_groups = OrderedDict({
    "All_40: Eye+Head": [
        "Convergence_distance",
        "Left_Eye_Openness", "Right_Eye_Openness",
        "LeftPupilDiameter", "RightPupilDiameter",
        "LeftPupilPosInSensorX", "LeftPupilPosInSensorY", "RightPupilPosInSensorX", "RightPupilPosInSensorY",
        "NrmSRLeftEyeGazeDirX", "NrmSRLeftEyeGazeDirY", "NrmSRLeftEyeGazeDirZ",
        "NrmSRRightEyeGazeDirX", "NrmSRRightEyeGazeDirY", "NrmSRRightEyeGazeDirZ",
        "GazeOriginLclSpc_X", "GazeOriginLclSpc_Y", "GazeOriginLclSpc_Z",
        "GazeOriginWrldSpc_X", "GazeOriginWrldSpc_Y", "GazeOriginWrldSpc_Z",
        "GazeDirectionLclSpc_X", "GazeDirectionLclSpc_Y", "GazeDirectionLclSpc_Z",
        "GazeDirectionWrldSpc_X", "GazeDirectionWrldSpc_Y", "GazeDirectionWrldSpc_Z",
        "NrmLeftEyeOriginX", "NrmLeftEyeOriginY", "NrmLeftEyeOriginZ",
        "NrmRightEyeOriginX", "NrmRightEyeOriginY", "NrmRightEyeOriginZ",
        "HeadQRotationX", "HeadQRotationY", "HeadQRotationZ", "HeadQRotationW",
        "HeadEulX", "HeadEulY", "HeadEulZ",
    ],
})

## Models
Every model standardizes the features inside the pipeline, so the scaler is fitted on the training folds only.

In [ ]:
def random_oversample(X, y, seed=SEED):
    ros = RandomOverSampler(random_state=seed)
    return ros.fit_resample(X, y)


def rf_pipeline():
    return make_pipeline(
        StandardScaler(),
        RandomForestClassifier(n_estimators=600, max_depth=10, min_samples_split=10, min_samples_leaf=4,
                               max_features="sqrt", random_state=SEED, n_jobs=-1),
    )


def et_pipeline():
    return make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(n_estimators=300, max_depth=12, min_samples_split=10, min_samples_leaf=4,
                             max_features=0.8, random_state=SEED, n_jobs=-1),
    )


def xgb_pipeline():
    return make_pipeline(
        StandardScaler(),
        XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                      gamma=0.1, reg_alpha=0.1, objective="multi:softprob", eval_metric="mlogloss",
                      random_state=SEED, n_jobs=-1),
    )


def build_models():
    return [
        ("SVM-RBF", make_pipeline(StandardScaler(),
                                  SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=SEED))),
        ("Random Forest", rf_pipeline()),
        ("XGBoost", xgb_pipeline()),
        ("Extra Trees", et_pipeline()),
        ("Stacking-Tree", make_pipeline(
            StackingClassifier(
                estimators=[("rf", rf_pipeline()), ("et", et_pipeline()), ("xgb", xgb_pipeline())],
                final_estimator=LogisticRegression(max_iter=2000, solver="lbfgs", random_state=SEED),
                stack_method="predict_proba",
                cv=5,
                n_jobs=-1,
            ))),
    ]

## Stratified 10-fold cross-validation (cross-user mixing)
Random oversampling is applied to the training folds only.

In [ ]:
def run_experiment(df_src, exp_name, feature_cols):
    df_local = df_src.dropna(subset=["cs"]).copy()
    y_all = df_local["cs"].astype(int).values
    X_all = df_local[feature_cols].to_numpy(dtype=float)
    dataset_name = "Prewindowed samples"
    labels = [0, 1, 2, 3]

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    results_folds, results_avg = [], []

    for model_name, model_proto in build_models():
        train_times, test_times = [], []
        fold_accuracies, fold_precisions, fold_recalls = [], [], []
        fold_f1_each_list, fold_f1_macro_list = [], []

        for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X_all, y_all), 1):
            X_train, y_train = X_all[train_idx], y_all[train_idx]
            X_test, y_test = X_all[test_idx], y_all[test_idx]

            model = clone(model_proto)
            last_step_name = list(model.named_steps.keys())[-1]
            estimator = model.named_steps[last_step_name]
            X_train, y_train = random_oversample(X_train, y_train)

            fit_kwargs = {}
            if not isinstance(estimator, StackingClassifier):
                classes = np.unique(y_train)
                weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
                class_weight_dict = {int(c): w for c, w in zip(classes, weights)}
                sample_weight = np.array([class_weight_dict[int(lbl)] for lbl in y_train])
                if "sample_weight" in signature(estimator.fit).parameters:
                    fit_kwargs[f"{last_step_name}__sample_weight"] = sample_weight

            t_train0 = time.time()
            model.fit(X_train, y_train, **fit_kwargs)
            train_time = time.time() - t_train0
            train_times.append(train_time)

            t_inf0 = perf_counter()
            y_pred = model.predict(X_test)
            inference_time_per_sample = (perf_counter() - t_inf0) / len(X_test)
            test_times.append(inference_time_per_sample)

            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, labels=labels, average=None, zero_division=0)
            rec = recall_score(y_test, y_pred, labels=labels, average=None, zero_division=0)
            f1_each = f1_score(y_test, y_pred, labels=labels, average=None, zero_division=0)
            f1_macro = f1_score(y_test, y_pred, labels=labels, average="macro", zero_division=0)
            fold_accuracies.append(acc)
            fold_precisions.append(prec)
            fold_recalls.append(rec)
            fold_f1_each_list.append(f1_each)
            fold_f1_macro_list.append(f1_macro)

            results_folds.append({
                "experiment": exp_name, "dataset": dataset_name, "model": model_name, "fold": fold_idx,
                "accuracy": round(acc, 4),
                **{f"precision {i}": round(prec[i], 4) for i in range(4)},
                **{f"recall {i}": round(rec[i], 4) for i in range(4)},
                **{f"f1_{i}": round(f1_each[i], 4) for i in range(4)},
                "f1_macro": round(f1_macro, 4),
                "train_time_sec": round(train_time, 4),
                "inference_time_s_per_sample": round(inference_time_per_sample, 8),
            })

        avg_prec = np.mean(fold_precisions, axis=0)
        avg_rec = np.mean(fold_recalls, axis=0)
        avg_f1_each = np.mean(np.vstack(fold_f1_each_list), axis=0)
        results_avg.append({
            "experiment": exp_name, "dataset": dataset_name, "model": model_name,
            "test accuracy": round(float(np.mean(fold_accuracies)), 4),
            **{f"test precision {i}": round(avg_prec[i], 4) for i in range(4)},
            **{f"test recall {i}": round(avg_rec[i], 4) for i in range(4)},
            **{f"test f1 {i}": round(avg_f1_each[i], 4) for i in range(4)},
            "test f1 macro": round(float(np.mean(fold_f1_macro_list)), 4),
            "time_train_mean_sec": round(float(np.mean(train_times)), 2),
            "mean_inf_time_s_per_sample": round(float(np.mean(test_times)), 8),
        })
        print(f"{model_name}: Acc={np.mean(fold_accuracies):.3f}, Macro F1={np.mean(fold_f1_macro_list):.3f}, "
              f"train={np.mean(train_times):.2f} s, inference={np.mean(test_times):.8f} s/sample")

    return pd.DataFrame(results_folds), pd.DataFrame(results_avg)

In [ ]:
all_folds, all_avg = [], []
for exp_name, fea_cols in exp_feature_groups.items():
    feature_cols = [c if c in df.columns else f"{c}__mean" for c in fea_cols]
    folds_df, avg_df = run_experiment(df, exp_name, feature_cols)
    all_folds.append(folds_df)
    all_avg.append(avg_df)

pd.concat(all_folds, ignore_index=True).to_csv(os.path.join(OUT_DIR, "ml_cv_results_folds.csv"), index=False)
pd.concat(all_avg, ignore_index=True).to_csv(os.path.join(OUT_DIR, "ml_cv_results_avg.csv"), index=False)